# Пилот синтетического эксперимента

Цели проекта — подготовить сбор данных к A/B-тесту, сравнить четыре модели для предварительного выбора и проверить платформу. Здесь проектируем выборку и смотрим по одному примеру каждого сценария.

## Выборка

1 000 пользователей, 90 дней. В первые 60 дней выходят восемь публикаций. Следующие 30 дней продолжаем наблюдать продажи. Два курса — «Аналитика старт» и «Аналитика про». Исторические суммы по позициям этих курсов служат основой чеков. Они могут уже включать скидки или доли пакета, поэтому исторический прайс из них не восстанавливаем.

Одинаковые пользователи, расписание, затраты и заранее сгенерированные случайные числа во всех сценариях. Касания к недоступному продукту в сценарии запуска исключаются. Каждый пост указывает один курс. Категории — обычная публикация, скидка, анонс события и запуск. В пилоте все восемь размещений условно платные, по 6 000 ₽. Тип публикации хранится отдельно от способа размещения.

## Четыре сценария

| Сценарий | Что меняется относительно ровного спроса |
|---|---|
| Ровный спрос | Оба курса доступны. Базовая вероятность покупки каждого курса — 0,1% на пользователя в день. После рекламного касания к этому курсу прибавляем 0,3 п.п. в течение семи дней, включая день касания. |
| Внешнее событие | Дни 31–37: вероятность фоновой покупки «Аналитика про» умножается на 3. Публикации и рекламная прибавка те же. |
| Запуск курса | «Аналитика про» доступен с дня 31. Дни 31–37: фон для этого курса умножается на 3. Пост дня 31 получает категорию «запуск». Дата запуска условная. |
| Нулевой эффект | Ровный спрос и то же расписание касаний. Рекламная прибавка и снижение цены отключены. |

## Предположения генератора

- Вероятность касания каждого доступного размещения 16%. После рекламного касания вероятность просмотра 65%, после просмотра вероятность заявки 25%. У 30% пользователей в среднем есть один заход без метки.
- Заявки и оплаты генерируются отдельно. Таблица ниже считает заявки пользователей, которые совсем не купили за период. Для сверки оплат добавлена условная связь с последней заявкой по одному из купленных курсов после предыдущего заказа.
- Не больше двух заказов у пользователя, интервал более трёх дней. Фоновая вероятность повторной покупки уменьшается в четыре раза. Два курса в одном заказе возможны с вероятностью 10% при доступности обоих. Сумма заказа равна сумме позиций.
- Снижение цены на 15% действует для покупателя с касанием к скидочному посту о соответствующем курсе за последние семь дней. Скидка применяется к выбранной исторической сумме как условный сценарий. Рекламная прибавка к вероятности покупки одинакова для всех категорий.
- Шаг генерации — день. Внутри дня публикация и касание считаются доступными к моменту покупки. В таблицах дополнительно заданы времена публикации 09:00, касания 10:00, просмотра 11:00, заявки 12:00 и оплаты 18:00 условного дня.
- Семидневный срок воздействия в генераторе и 30-дневное окно атрибуции — разные параметры. Скрытая метка заказа описывает ветку генерации. В многокурсовом заказе она относится к покупке, инициировавшей заказ. Все вероятности, затраты и величины всплесков заданы для пилота.



In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/base.csv').exists())
sys.path.insert(0, str(ROOT / 'src'))
from experiments.scenario_pilot import generate_pilots, summarize

pilots = dict(generate_pilots(ROOT / 'data/raw/base.csv', seed=42, n_users=1000))
summary = summarize(pilots.items())
summary


,scenario,users,visitors,touches,views,leads,leads_from_nonbuyers,buyers,orders,items,revenue_rub,ad_tagged_orders
0,flat,1000,835,1623,828,205,167,182,185,209,1542859.75,23
1,external_event,1000,835,1623,828,205,161,197,202,225,1672171.25,26
2,course_launch,1000,807,1458,715,176,142,162,166,184,1362644.75,17
3,zero_ad_effect,1000,835,1623,828,205,172,163,166,186,1384796.50,0


Один прогон помогает проверить объём, долю покупателей, наличие заявок без покупок и состав заказов. Для проверки календаря смотрим продажи по курсам и дням. Различия между четырьмя пилотами описывают условия симуляции. Следующий этап — проверка расчётов и отчёта бота, затем сравнение моделей на повторных прогонах.


In [2]:
import pandas as pd

daily_course_items = pd.concat([
    tables['items'].merge(tables['orders'][['order_id', 'day']], on='order_id', validate='many_to_one')
    .groupby(['day', 'course']).size().rename('items').reset_index().assign(scenario=name)
    for name, tables in pilots.items()
], ignore_index=True)
daily_course_items


,day,course,items,scenario
0,1,Аналитика старт,2,flat
1,2,Аналитика старт,2,flat
2,3,Аналитика про,1,flat
3,4,Аналитика старт,3,flat
4,5,Аналитика про,1,flat
...,...,...,...,...
455,88,Аналитика старт,1,zero_ad_effect
456,89,Аналитика про,1,zero_ad_effect
457,89,Аналитика старт,1,zero_ad_effect
458,90,Аналитика про,1,zero_ad_effect


## История и единица покупки

Строки одного студента с одинаковым точным временем образуют один заказ. Разные курсы внутри заказа остаются отдельными позициями с исходными суммами. Наличие нескольких курсов само по себе не подтверждает акцию. История используется для проверки единицы покупки и калибровки сумм. Атрибуцию запускаем на синтетике, где есть касания.


In [3]:
import numpy as np
import pandas as pd

from core.attribution import MODELS, compute
from core.importer import read_sales

history_items = read_sales(ROOT / 'data/raw/base.csv')
history_orders = history_items.groupby(['student_id', 'ts'], as_index=False).agg(
    amount=('amount', 'sum'), items=('course', 'size'),
    courses=('course', 'nunique'),
)
assert np.isclose(history_items.amount.sum(), history_orders.amount.sum(), atol=.005, rtol=0)
pd.Series({
    'Строки истории': len(history_items),
    'Заказы': len(history_orders),
    'Заказы с несколькими курсами': int(history_orders.courses.gt(1).sum()),
    'Выручка, ₽': round(history_orders.amount.sum(), 2),
})

Строки истории                      795.00
Заказы                              628.00
Заказы с несколькими курсами        153.00
Выручка, ₽                      5904671.67


## Прогон четырёх моделей

Синтетический день переводим в условную дату. Касания ставим на 10:00, оплаты на 18:00. Так сохраняем порядок, заданный генератором, включая покупку в день касания. Несколько касаний в одно время движок упорядочивает по идентификатору. Внутри каждого сценария все модели получают одну выборку и окно 30 дней.


In [4]:
def at_hour(days, hour):
    return pd.Timestamp('2025-01-01') + pd.to_timedelta(days - 1, unit='D') + pd.Timedelta(hours=hour)

def checked_attribution(touches, orders, model):
    result = compute(touches, orders, model=model, window_days=30)
    totals = result.groupby('order_id').agg(
        weight=('weight', 'sum'), revenue=('attributed_revenue', 'sum'),
    ).reindex(orders.order_id)
    np.testing.assert_allclose(totals.weight, 1, rtol=0, atol=1e-10)
    np.testing.assert_allclose(totals.revenue, orders.amount, rtol=0, atol=.005)

    matched = result.dropna(subset=['touch_id']).merge(
        touches[['touch_id', 'timestamp']], on='touch_id',
    ).merge(orders[['order_id', 'timestamp', 'previous']], on='order_id', suffixes=('_touch', '_order'))
    age = matched.timestamp_order - matched.timestamp_touch
    assert age.between(pd.Timedelta(0), pd.Timedelta(days=30)).all()
    assert (matched.previous.isna() | matched.timestamp_touch.gt(matched.previous)).all()
    return result

checks = []
placements_results = []
for scenario, tables in pilots.items():
    touches = tables['touches'].rename(columns={'user_id': 'user_id_hash'}).assign(
        timestamp=at_hour(tables['touches'].day, 10),
    )
    orders = tables['orders'].rename(columns={'user_id': 'user_id_hash'}).assign(
        timestamp=at_hour(tables['orders'].day, 18), variable_costs=0,
    ).sort_values(['user_id_hash', 'timestamp'])
    orders['previous'] = orders.groupby('user_id_hash').timestamp.shift()
    assert not orders.duplicated(['user_id_hash', 'timestamp']).any()
    item_totals = tables['items'].groupby('order_id').amount.sum().reindex(orders.order_id)
    np.testing.assert_allclose(item_totals, orders.amount, rtol=0, atol=.005)

    # Independent lookup of orders with at least one eligible placement touch.
    candidates = orders.merge(touches.dropna(subset=['placement_id']), on='user_id_hash',
                              suffixes=('_order', '_touch'))
    eligible = candidates.timestamp_touch.between(
        candidates.timestamp_order - pd.Timedelta(days=30), candidates.timestamp_order,
    ) & (candidates.previous.isna() | candidates.timestamp_touch.gt(candidates.previous))
    expected_unassigned = set(orders.order_id) - set(candidates.loc[eligible, 'order_id'])

    for model in MODELS:
        result = checked_attribution(touches, orders, model)
        unassigned = result.placement_id.isna()
        assert set(result.loc[unassigned, 'order_id']) == expected_unassigned
        revenue = result.groupby('placement_id').attributed_revenue.sum()
        placements = tables['placements'][['placement_id', 'cost']].copy()
        placements['revenue'] = placements.placement_id.map(revenue).fillna(0)
        placements['romi_pct'] = 100 * (placements.revenue - placements.cost) / placements.cost
        placements_results.append(placements.assign(scenario=scenario, model=model))
        checks.append({
            'scenario': scenario, 'model': model, 'orders': len(orders),
            'repeat_orders': int(orders.previous.notna().sum()),
            'orders_without_eligible_touch': int(unassigned.sum()),
            'total_rub': round(result.attributed_revenue.sum(), 2),
            'assigned_rub': round(placements.revenue.sum(), 2),
        })

checks = pd.DataFrame(checks)
placements_results = pd.concat(placements_results, ignore_index=True)
checks

          scenario     model  orders  repeat_orders  orders_without_eligible_touch   total_rub  assigned_rub
0             flat      last     185              3                            106  1542859.75     673873.25
1             flat     first     185              3                            106  1542859.75     673873.25
2             flat    linear     185              3                            106  1542859.75     673873.25
3             flat  position     185              3                            106  1542859.75     673873.25
4   external_event      last     202              5                            113  1672171.25     763824.75
5   external_event     first     202              5                            113  1672171.25     763824.75
6   external_event    linear     202              5                            113  1672171.25     763824.75
7   external_event  position     202              5                            113  1672171.25     763824.75
8    course_launch 

## Результаты

В сценарии с нулевым эффектом выключены и рекламная прибавка к вероятности покупки, и скидки. Выручка всё равно распределяется между размещениями, если перед покупкой были подходящие касания. Сравнение показывает чувствительность отчёта к модели на заданной выборке.


In [5]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 11, 'svg.fonttype': 'none'})
chart_dir = ROOT / 'outputs/experiments/scenario_pilot'
chart_dir.mkdir(parents=True, exist_ok=True)

scenario_names = {
    'flat': 'Ровный спрос', 'external_event': 'Внешнее событие',
    'course_launch': 'Запуск курса', 'zero_ad_effect': 'Нулевой эффект рекламы',
}
for metric, label in [('revenue', 'Приписанная выручка, ₽'), ('romi_pct', 'ROMI, %')]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True, layout='constrained')
    for ax, (scenario, name) in zip(axes.flat, scenario_names.items()):
        rows = placements_results.loc[placements_results.scenario.eq(scenario)]
        rows.pivot(index='placement_id', columns='model', values=metric)[list(MODELS)].plot.bar(
            ax=ax, legend=False, rot=0,
        )
        if metric == 'romi_pct':
            ax.axhline(0, color='gray', linewidth=1)
        ax.set(title=name, xlabel='Размещение', ylabel=label)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside upper center', ncol=4)
    fig.suptitle('Синтетика · 1 000 пользователей · 90 дней · seed 42', y=1.12)
    fig.savefig(chart_dir / f'{metric}.svg', bbox_inches='tight')
    fig.savefig(chart_dir / f'{metric}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Состав выборки и продукты
Все доли пользователей рассчитаны от 1 000 человек. Среднее по посетителям включает только людей с касанием. Касания считаются за весь период, а пути покупки — в окне атрибуции после предыдущей покупки. Продажа курса означает позицию заказа.


In [6]:
# Описательная статистика для всех пользователей, включая нулевые касания.
from cycler import cycler
plt.rcParams.update({'axes.prop_cycle': cycler(color=['#4C72B0', '#DD8452', '#8172B3', '#55A868']),
                     'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

profiles = []
user_frames = {}
product_frames = []
for scenario, tables in pilots.items():
    users = tables['users'].set_index('user_id').copy()
    users['touches'] = tables['touches'].groupby('user_id').size().reindex(users.index, fill_value=0)
    users['ad_touches'] = tables['touches'].dropna(subset=['placement_id']).groupby('user_id').size().reindex(users.index, fill_value=0)
    users['orders'] = tables['orders'].groupby('user_id').size().reindex(users.index, fill_value=0)
    user_frames[scenario] = users
    profiles.append({'scenario': scenario, 'buyers_pct': 100 * users.orders.gt(0).mean(),
                     'mean_touches_all': users.touches.mean(),
                     'mean_touches_visitors': users.loc[users.touches.gt(0), 'touches'].mean(),
                     'mean_ad_touches_all': users.ad_touches.mean(),
                     'zero_touches_pct': 100 * users.touches.eq(0).mean()})
    items = tables['items'].merge(tables['orders'][['order_id', 'user_id', 'day']], on='order_id', validate='many_to_one')
    product_frames.append(items.assign(scenario=scenario))
profile = pd.DataFrame(profiles).set_index('scenario')
product_items = pd.concat(product_frames, ignore_index=True)
assert product_items.groupby('scenario').amount.sum().round(2).equals(summary.set_index('scenario').revenue_rub.sort_index().round(2))
print(profile.round(3).to_string())

def save_sample_chart(fig, name):
    fig.savefig(chart_dir / f'{name}.svg', bbox_inches='tight')
    fig.savefig(chart_dir / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

names = list(scenario_names.values())
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout='constrained')
for ax, column, title in zip(axes, ['buyers_pct', 'zero_touches_pct'],
                           ['Купили за 90 дней', 'Не было ни одного касания за 90 дней']):
    bars = ax.barh(names, profile[column], color='#4C72B0')
    ax.bar_label(bars, labels=[f'{v:.1f}%' for v in profile[column]], padding=5)
    ax.set(xlim=(0, 100), title=title, xlabel='Доля от всех 1 000 пользователей, %')
    ax.invert_yaxis()
fig.suptitle('Состав синтетической выборки · seed 42')
save_sample_chart(fig, 'sample_population')

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True, layout='constrained')
max_touches = max(frame.touches.max() for frame in user_frames.values())
for ax, (scenario, frame) in zip(axes.flat, user_frames.items()):
    counts = frame.touches.value_counts().reindex(range(max_touches + 1), fill_value=0).sort_index()
    ax.bar(counts.index, counts / len(frame) * 100, color='#4C72B0')
    ax.set(title=f'{scenario_names[scenario]}\nСреднее {frame.touches.mean():.2f}; медиана {frame.touches.median():.0f}',
           xlabel='Число касаний за 90 дней', ylabel='Доля всех пользователей, %', xticks=range(max_touches + 1))
    ax.tick_params(axis='x', labelbottom=True)
fig.suptitle('Все заходы в бота, включая заходы без метки · seed 42')
save_sample_chart(fig, 'sample_touches')

fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for ax, metric, title in zip(axes, ['count', 'sum'], ['Проданные позиции, шт.', 'Выручка по позициям, ₽']):
    values = product_items.groupby(['scenario', 'course']).amount.agg(metric).unstack().reindex(scenario_names)
    values.index = names
    values.plot.barh(ax=ax, width=.7)
    ax.set(title=title, ylabel='')
    ax.invert_yaxis()
    ax.legend(title='Курс', loc='upper center', bbox_to_anchor=(.5, -.10), ncol=1)
fig.suptitle('Два курса в синтетической выборке · позиции одного заказа считаются отдельно')
save_sample_chart(fig, 'sample_products')

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True, layout='constrained')
for ax, (scenario, name) in zip(axes.flat, scenario_names.items()):
    items = product_items.loc[product_items.scenario.eq(scenario)]
    daily = items.groupby(['day', 'course']).size().unstack(fill_value=0).reindex(range(1, 91), fill_value=0)
    for course, style in zip(daily.columns, ['-', '--']):
        ax.plot(daily.index, daily[course], label=course, linestyle=style, linewidth=1.4)
    if scenario in ('external_event', 'course_launch'):
        ax.axvspan(31, 37, color='#8172B3', alpha=.15, label='Заданный рост спроса, дни 31–37')
    if scenario == 'course_launch':
        ax.axvline(31, color='#555555', linestyle=':', label='Доступность нового курса')
    ax.set(title=name, xlabel='День симуляции', ylabel='Проданные позиции за день, шт.')
    ax.legend(fontsize=8)
fig.suptitle('Продажи конкретных курсов во времени · один прогон, seed 42')
save_sample_chart(fig, 'sample_daily_products')

# Проверка того, сколько покупок действительно различают модели.
path_rows = []
for scenario, tables in pilots.items():
    orders = tables['orders'].sort_values(['user_id', 'day']).copy()
    orders['previous_day'] = orders.groupby('user_id').day.shift()
    merged = orders.merge(tables['touches'].dropna(subset=['placement_id']), on='user_id', suffixes=('_order', '_touch'))
    # Время оплаты 18:00 и касания 10:00 даёт возраст (разница дней + 8 часов).
    valid = (merged.day_touch.le(merged.day_order) & merged.day_touch.gt(merged.day_order - 30)
             & (merged.previous_day.isna() | merged.day_touch.gt(merged.previous_day)))
    eligible = merged.loc[valid]
    per_order = eligible.groupby('order_id').size().reindex(orders.order_id, fill_value=0)
    unique_posts = eligible.groupby('order_id').placement_id.nunique().reindex(orders.order_id, fill_value=0)
    path_rows.append({'scenario': scenario, 'no_touch_pct': 100 * per_order.eq(0).mean(),
                     'one_touch_pct': 100 * per_order.eq(1).mean(),
                     'multiple_touches_pct': 100 * per_order.ge(2).mean(),
                     'multiple_placements_orders': int(unique_posts.ge(2).sum())})
paths = pd.DataFrame(path_rows).set_index('scenario')
fig, ax = plt.subplots(figsize=(11, 4), layout='constrained')
plot = paths[['no_touch_pct', 'one_touch_pct', 'multiple_touches_pct']].copy()
plot.columns = ['0 касаний', '1 касание', '2 и более касаний']
plot.index = names
plot.plot.barh(stacked=True, ax=ax)
ax.set(xlim=(0, 100), xlabel='Доля заказов, %', ylabel='', title='Касания, подходящие для атрибуции каждой покупки')
ax.invert_yaxis()
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.18), ncol=3)
save_sample_chart(fig, 'sample_paths')
print(paths.round(2).to_string())

profile.to_csv(chart_dir / 'sample_profile.csv')
paths.to_csv(chart_dir / 'sample_paths.csv')
product_items.groupby(['scenario', 'course']).agg(items=('order_id', 'size'), revenue=('amount', 'sum')).to_csv(chart_dir / 'sample_products.csv')


                buyers_pct  mean_touches_all  mean_touches_visitors  mean_ad_touches_all  zero_touches_pct
scenario                                                                                                  
flat                  18.2             1.623                  1.944                1.303              16.5
external_event        19.7             1.623                  1.944                1.303              16.5
course_launch         16.2             1.458                  1.807                1.138              19.3
zero_ad_effect        16.3             1.623                  1.944                1.303              16.5
                no_touch_pct  one_touch_pct  multiple_touches_pct  multiple_placements_orders
scenario                                                                                     
flat                   57.30          37.84                  4.86                           9
external_event         55.94          38.12                  5.94           

## Время, заявки и повторные прогоны
События получают условное время в МСК. Для заказа выбирается последняя заявка того же пользователя по одному из купленных курсов, созданная после предыдущей покупки и до оплаты. При отсутствии такой заявки связь остаётся пустой. Это допущение симуляции, а не восстановление причины покупки. Покупки и вероятности генератора сохраняются.
Проверяем пять seed и выводим диапазоны, среднее и стандартное отклонение. Это разброс симуляции. Историческая подвыборка двух курсов исключает суммы остальных курсов даже из совместных заказов. Длительность истории берётся между первой и последней продажей и не доказывает полноту наблюдения.


In [7]:
# Пять повторов характеризуют чувствительность к случайности генератора.
seeds = [42, 43, 44, 45, 46]
run_rows, ranking_rows = [], []
for seed in seeds:
    sample = pilots if seed == 42 else dict(generate_pilots(ROOT / 'data/raw/base.csv', seed=seed))
    for scenario, tables in sample.items():
        orders = tables['orders'].rename(columns={'user_id': 'user_id_hash'}).assign(variable_costs=0).sort_values(['user_id_hash', 'timestamp'])
        orders['previous'] = orders.groupby('user_id_hash').timestamp.shift()
        touches = tables['touches'].rename(columns={'user_id': 'user_id_hash'})
        linked = orders.dropna(subset=['lead_id']).merge(tables['leads'], on='lead_id', suffixes=('_order', '_lead'), validate='one_to_one')
        assert linked.user_id_hash.eq(linked.user_id).all()
        assert linked.timestamp_lead.le(linked.timestamp_order).all()
        assert (linked.previous.isna() | linked.timestamp_lead.gt(linked.previous)).all()
        assert not orders.duplicated(['user_id_hash', 'timestamp']).any()
        paid_lead_ids = set(orders.lead_id.dropna())
        run_rows.append({'seed': seed, 'scenario': scenario, 'orders': len(orders),
                         'buyers': orders.user_id_hash.nunique(), 'revenue': orders.amount.sum(),
                         'linked_orders': len(linked),
                         'leads_without_linked_order': int((~tables['leads'].lead_id.isin(paid_lead_ids)).sum())})
        for model in MODELS:
            result = checked_attribution(touches, orders, model)
            revenue = result.groupby('placement_id').attributed_revenue.sum()
            placements = tables['placements'][['placement_id', 'cost']].copy()
            placements['revenue'] = placements.placement_id.map(revenue).fillna(0)
            placements['romi_pct'] = 100 * (placements.revenue - placements.cost) / placements.cost
            placements['rank'] = placements.romi_pct.rank(ascending=False, method='average')
            ranking_rows.append(placements.assign(seed=seed, scenario=scenario, model=model))
runs = pd.DataFrame(run_rows)
rankings = pd.concat(ranking_rows, ignore_index=True)
run_summary = runs.groupby('scenario').agg(
    orders_min=('orders', 'min'), orders_mean=('orders', 'mean'), orders_max=('orders', 'max'),
    buyers_min=('buyers', 'min'), buyers_max=('buyers', 'max'),
    revenue_mean=('revenue', 'mean'), revenue_std=('revenue', 'std'),
    linked_orders_mean=('linked_orders', 'mean'),
    leads_without_linked_order_mean=('leads_without_linked_order', 'mean'),
)
rank_summary = rankings.groupby(['scenario', 'model', 'placement_id']).agg(
    rank_min=('rank', 'min'), rank_max=('rank', 'max'), rank_mean=('rank', 'mean'),
    romi_mean=('romi_pct', 'mean'), romi_std=('romi_pct', 'std'),
)
runs.to_csv(chart_dir / 'seed_runs.csv', index=False)
run_summary.to_csv(chart_dir / 'seed_summary.csv')
rank_summary.to_csv(chart_dir / 'rank_summary.csv')
print(run_summary.round(2).to_string())
print('\nСвязи заявок в основном прогоне:')
print(runs.loc[runs.seed.eq(42), ['scenario', 'linked_orders', 'leads_without_linked_order']].to_string(index=False))

# Историческая подвыборка включает только суммы двух выбранных курсов.
history_days = (history_items.ts.max().normalize() - history_items.ts.min().normalize()).days + 1
comparison_rows = []
for label, items in [('Вся история', history_items), ('История двух курсов', history_items.loc[history_items.course.isin(['Аналитика старт', 'Аналитика про'])])]:
    orders = items.groupby(['student_id', 'ts']).agg(amount=('amount', 'sum'), items=('course', 'size')).reset_index()
    counts = orders.groupby('student_id').size()
    comparison_rows.append({'sample': label, 'days': history_days, 'orders': len(orders),
                            'orders_per_day': len(orders)/history_days, 'mean_amount': orders.amount.mean(),
                            'median_amount': orders.amount.median(), 'repeat_buyers_pct': 100*counts.ge(2).mean(),
                            'multiple_items_pct': 100*orders['items'].ge(2).mean()})
for scenario, tables in pilots.items():
    orders = tables['orders']; counts = orders.groupby('user_id').size()
    comparison_rows.append({'sample': scenario, 'days': 90, 'orders': len(orders),
                            'orders_per_day': len(orders)/90, 'mean_amount': orders.amount.mean(),
                            'median_amount': orders.amount.median(), 'repeat_buyers_pct': 100*counts.ge(2).mean(),
                            'multiple_items_pct': 100*tables['items'].groupby('order_id').size().ge(2).mean()})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(chart_dir / 'history_comparison.csv', index=False)
print('\nСопоставление с историей:')
print(comparison.round(2).to_string(index=False))


                orders_min  orders_mean  orders_max  buyers_min  buyers_max  revenue_mean  revenue_std  linked_orders_mean  leads_without_linked_order_mean
scenario                                                                                                                                                   
course_launch          155        171.0         183         154         178     1376778.8     89207.55                14.0                            166.4
external_event         180        202.8         218         179         211     1676248.1    141411.26                16.6                            191.6
flat                   169        188.0         205         169         201     1554114.8    138589.76                16.0                            192.2
zero_ad_effect         141        163.2         184         141         183     1355876.9    172786.62                10.2                            198.0

Связи заявок в основном прогоне:
      scenario  linked_orders 